### 1 · Data Preprocessing — Heading-Aware Chunking with Enriched Metadata

This notebook takes raw Confluence markdown docs and produces **retrieval-optimized
chunks** with clean, structured metadata.

### Pipeline
```
data/raw/confluence/**/*.md
  → Load each file as raw text
  → Extract front matter (title, version, owner, audience, status)
  → Derive doc_type from folder path
  → Split by markdown headings (section-aware)
  → Enrich each chunk with semantic metadata
  → Drop noisy fields
  → Export to data/processed/
```

In [14]:
import re
import json
from pathlib import Path
from pprint import pprint

from langchain_core.documents import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)
from dotenv import load_dotenv

load_dotenv(override=True)

RAW_DIR = Path("../data/raw/confluence")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Source docs: {list(RAW_DIR.rglob('*.md'))}")

Source docs: [PosixPath('../data/raw/confluence/index.md'), PosixPath('../data/raw/confluence/features/journey-canvas.md'), PosixPath('../data/raw/confluence/features/action-builder.md'), PosixPath('../data/raw/confluence/features/history-service.md'), PosixPath('../data/raw/confluence/operations/code-quality-audit-plan.md'), PosixPath('../data/raw/confluence/architecture/firebase.md'), PosixPath('../data/raw/confluence/architecture/code-quality.md'), PosixPath('../data/raw/confluence/architecture/overview.md'), PosixPath('../data/raw/confluence/architecture/theming.md'), PosixPath('../data/raw/confluence/architecture/redux.md')]


### Step 1 — Extract Front Matter & Derive Doc Type

Each markdown file has a title on line 1 (`# Title`) and optional front matter
(`> **Version:** 1.0`, `> **Owner:** ...`). We parse these into structured fields.

The `doc_type` is derived from the folder path:
- `confluence/architecture/*.md` → `"architecture"`
- `confluence/features/*.md` → `"feature"`
- `confluence/operations/*.md` → `"operations"`
- `confluence/business/*.md` → `"business"`
- `confluence/index.md` → `"index"`

In [15]:
def extract_front_matter(text: str) -> dict:
    """Parse structured fields from the top of a markdown file.

    Handles two common patterns in our docs:
      1.  # Title on line 1
      2.  > **Key:** Value   (blockquote front matter)
      3.  **Key:** Value     (bold key-value on early lines)
    """
    meta = {}

    # 1) Document title — first H1
    title_match = re.search(r"^#\s+(.+)$", text, re.MULTILINE)
    if title_match:
        meta["doc_title"] = title_match.group(1).strip()

    # 2) Blockquote front matter:  > **Version:** 1.0
    for m in re.finditer(r">\s*\*\*(.+?):\*\*\s*(.+)", text):
        key = m.group(1).strip().lower().replace(" ", "_")
        meta[key] = m.group(2).strip()

    # 3) Bold key-value (non-blockquote):  **Last updated:** 2026-03-18
    for m in re.finditer(r"(?<!>)\s*\*\*(.+?):\*\*\s*(.+)", text):
        key = m.group(1).strip().lower().replace(" ", "_")
        if key not in meta:  # don't overwrite blockquote matches
            meta[key] = m.group(2).strip()

    return meta


# ── Derive doc_type from folder path ────────────────────────────────────────

DOC_TYPE_MAP = {
    "architecture": "architecture",
    "features": "feature",
    "operations": "operations",
    "business": "business",
}

def derive_doc_type(file_path: Path, base_dir: Path) -> str:
    """Map folder name to a doc_type label."""
    relative = file_path.relative_to(base_dir)
    parts = relative.parts  # e.g. ('architecture', 'overview.md')
    if len(parts) >= 2:
        folder = parts[0]
        return DOC_TYPE_MAP.get(folder, "general")
    return "index"  # top-level files like index.md


# ── Quick test ───────────────────────────────────────────────────────────────

sample_path = RAW_DIR / "architecture" / "overview.md"
sample_text = sample_path.read_text()

print("=== Extracted front matter ===")
fm = extract_front_matter(sample_text)
pprint(fm)

print(f"\n=== Derived doc_type ===")
print(derive_doc_type(sample_path, RAW_DIR))

=== Extracted front matter ===
{'doc_title': 'AMS Admin Tool — Architecture Documentation',
 'last_updated': 'March 2026',
 'owner': 'Admin Tool Engineering',
 'status': 'Production',
 'version': '1.0'}

=== Derived doc_type ===
architecture


## Step 2 — Heading-Aware Markdown Splitting

We use **two splitters in sequence**:

1. `MarkdownHeaderTextSplitter` — splits by `#`, `##`, `###` headings and captures the
   heading hierarchy as metadata. This gives us **section-aware chunks**.
2. `RecursiveCharacterTextSplitter` — if any section is still too large (>1 000 chars),
   it gets sub-split while **preserving the heading metadata**.

This is better than `UnstructuredMarkdownLoader` because:
- Chunks align to logical sections, not arbitrary character windows
- Each chunk knows which heading path it belongs to
- No noisy element-level metadata (`parent_id`, `element_id`, etc.)

In [18]:
# ── Heading-aware splitter config ────────────────────────────────────────────

HEADERS_TO_SPLIT_ON = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=HEADERS_TO_SPLIT_ON,
    strip_headers=False,  # keep heading text in the chunk content
)

# Secondary splitter for oversized sections
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)


def split_markdown_file(file_path: Path, base_dir: Path) -> list[Document]:
    """Load a markdown file, split by headings, then sub-split large sections.

    Returns a list of Documents with enriched metadata:
      - doc_title, doc_type, audience, owner, status, version  (from front matter)
      - section, heading_hierarchy                              (from headings)
      - chunk_index                                             (position in doc)
      - source, last_modified                                   (from file)
    """
    raw_text = file_path.read_text(encoding="utf-8")
    stat = file_path.stat()

    # ── Extract document-level metadata ──────────────────────────────────
    front_matter = extract_front_matter(raw_text)
    doc_type = derive_doc_type(file_path, base_dir)
    source = str(file_path.relative_to(base_dir))

    doc_meta = {
        "source": source,
        "doc_title": front_matter.get("doc_title", file_path.stem),
        "doc_type": doc_type,
        "audience": front_matter.get("audience", "all").lower(),
        "owner": front_matter.get("owner", ""),
        "status": front_matter.get("status", "").lower(),
        "version": front_matter.get("version", ""),
        "last_modified": front_matter.get(
            "last_updated",
            # fallback to file mtime
            __import__("datetime").datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d"),
        ),
    }

    # ── Step 1: Split by headings ────────────────────────────────────────
    header_chunks = md_splitter.split_text(raw_text)

    # ── Step 2: Sub-split oversized sections ─────────────────────────────
    final_chunks = text_splitter.split_documents(header_chunks)

    # ── Step 3: Enrich each chunk with doc-level + section metadata ──────
    enriched = []
    for i, chunk in enumerate(final_chunks):
        # Build heading hierarchy from the header metadata
        heading_hierarchy = []
        for level in ["h1", "h2", "h3"]:
            if level in chunk.metadata:
                heading_hierarchy.append(chunk.metadata[level])

        # The most specific heading is the "section"
        section = heading_hierarchy[-1] if heading_hierarchy else ""

        # Merge doc-level + chunk-level metadata
        chunk.metadata = {
            **doc_meta,
            "section": section,
            "heading_hierarchy": heading_hierarchy,
            "chunk_index": i,
        }
        enriched.append(chunk)

    return enriched


# Quick test on one file
test_chunks = split_markdown_file(RAW_DIR / "architecture" / "overview.md", RAW_DIR)
print(f"Chunks from overview.md: {len(test_chunks)}")
print(f"\n=== Sample chunk (index 3) ===")
print(f"Content preview: {test_chunks[3].page_content[:200]}...")
print(f"\nMetadata:")
pprint(test_chunks[3].metadata)

Chunks from overview.md: 40

=== Sample chunk (index 3) ===
Content preview: | **Canvas** | React Flow (@xyflow/react) | 12.10 | Visual node-and-edge flow graph |
| **Layout** | dagre (@dagrejs/dagre) | 2.0 | Automatic graph layout |
| **Code Editor** | CodeMirror 6 | 6.0 | Gr...

Metadata:
{'audience': 'all',
 'chunk_index': 3,
 'doc_title': 'AMS Admin Tool — Architecture Documentation',
 'doc_type': 'architecture',
 'heading_hierarchy': ['AMS Admin Tool — Architecture Documentation',
                       'Tech Stack'],
 'last_modified': 'March 2026',
 'owner': 'Admin Tool Engineering',
 'section': 'Tech Stack',
 'source': 'architecture/overview.md',
 'status': 'production',
 'version': '1.0'}


## Step 3 — Process All Documents

Run the heading-aware splitter on every `.md` file in `data/raw/confluence/`.

In [19]:
# ── Process all markdown files ────────────────────────────────────────────────

all_chunks: list[Document] = []

md_files = sorted(RAW_DIR.rglob("*.md"))
print(f"Found {len(md_files)} markdown files\n")

for file_path in md_files:
    chunks = split_markdown_file(file_path, RAW_DIR)
    all_chunks.extend(chunks)
    print(f"  {str(file_path.relative_to(RAW_DIR)):<50} → {len(chunks):>3} chunks")

print(f"\n{'='*60}")
print(f"Total chunks: {len(all_chunks)}")

Found 10 markdown files

  architecture/code-quality.md                       →  28 chunks
  architecture/firebase.md                           →  32 chunks
  architecture/overview.md                           →  40 chunks
  architecture/redux.md                              →  20 chunks
  architecture/theming.md                            →  23 chunks
  features/action-builder.md                         →  41 chunks
  features/history-service.md                        →  38 chunks
  features/journey-canvas.md                         →  39 chunks
  index.md                                           →   7 chunks
  operations/code-quality-audit-plan.md              →  24 chunks

Total chunks: 292


## Step 4 — Before vs After: Metadata Comparison

Side-by-side comparison showing how metadata improves from raw Unstructured output to our enriched format.

In [20]:
# ── Before vs After ──────────────────────────────────────────────────────────

print("=" * 70)
print("BEFORE (raw Unstructured metadata)")
print("=" * 70)
old_metadata = {
    "source": "../data/raw/confluence/index.md",
    "emphasized_text_contents": ["Last updated:", "Audience:"],
    "emphasized_text_tags": ["b", "b"],
    "languages": ["eng"],
    "file_directory": "../data/raw/confluence",
    "filename": "index.md",
    "filetype": "text/markdown",
    "last_modified": "2026-03-19T13:19:16",
    "parent_id": "f33b733c1a0a83258c8bc68973982c09",
    "category": "UncategorizedText",
    "element_id": "1b2995d4420532d81884e3799cc3965a",
}
pprint(old_metadata)

print(f"\n{'=' * 70}")
print("AFTER (enriched, heading-aware metadata)")
print("=" * 70)
# Pick a representative chunk
sample = next(c for c in all_chunks if "architecture" in c.metadata.get("source", ""))
pprint(sample.metadata)

print(f"\n{'=' * 70}")
print("WHAT CHANGED")
print("=" * 70)
print("""
 ❌ REMOVED (noise):
    emphasized_text_contents, emphasized_text_tags, languages,
    file_directory, filename, filetype, parent_id, element_id, category

 ✅ ADDED (semantic):
    doc_title      → know which document this chunk is from
    doc_type       → filter by architecture / feature / operations at query time
    section        → which heading this chunk falls under
    heading_hierarchy → full breadcrumb path (H1 > H2 > H3)
    chunk_index    → reconstruct reading order
    audience       → scope answers for developers vs business
    owner          → attribution
    status         → skip deprecated docs
    version        → doc version tracking
""")

BEFORE (raw Unstructured metadata)
{'category': 'UncategorizedText',
 'element_id': '1b2995d4420532d81884e3799cc3965a',
 'emphasized_text_contents': ['Last updated:', 'Audience:'],
 'emphasized_text_tags': ['b', 'b'],
 'file_directory': '../data/raw/confluence',
 'filename': 'index.md',
 'filetype': 'text/markdown',
 'languages': ['eng'],
 'last_modified': '2026-03-19T13:19:16',
 'parent_id': 'f33b733c1a0a83258c8bc68973982c09',
 'source': '../data/raw/confluence/index.md'}

AFTER (enriched, heading-aware metadata)
{'audience': 'all',
 'chunk_index': 0,
 'doc_title': 'Code Quality & Style Guide for Developers',
 'doc_type': 'architecture',
 'heading_hierarchy': ['Code Quality & Style Guide for Developers'],
 'last_modified': 'March 2026',
 'owner': '',
 'section': 'Code Quality & Style Guide for Developers',
 'source': 'architecture/code-quality.md',
 'status': '',
 'version': ''}

WHAT CHANGED

 ❌ REMOVED (noise):
    emphasized_text_contents, emphasized_text_tags, languages,
    file_

## Step 5 — Chunk Quality Stats

Verify chunk sizes, section coverage, and metadata completeness before exporting.

In [21]:
# ── Chunk quality stats ──────────────────────────────────────────────────────

chunk_sizes = [len(c.page_content) for c in all_chunks]
sections_filled = sum(1 for c in all_chunks if c.metadata.get("section"))
titles_filled = sum(1 for c in all_chunks if c.metadata.get("doc_title"))
hierarchies_filled = sum(1 for c in all_chunks if c.metadata.get("heading_hierarchy"))

print("📊 Chunk Quality Report")
print("=" * 50)
print(f"  Total chunks:           {len(all_chunks)}")
print(f"  Min chunk size:         {min(chunk_sizes)} chars")
print(f"  Max chunk size:         {max(chunk_sizes)} chars")
print(f"  Avg chunk size:         {sum(chunk_sizes) // len(chunk_sizes)} chars")
print(f"  Median chunk size:      {sorted(chunk_sizes)[len(chunk_sizes)//2]} chars")
print()
print("📋 Metadata Completeness")
print("=" * 50)
print(f"  doc_title filled:       {titles_filled}/{len(all_chunks)} ({100*titles_filled//len(all_chunks)}%)")
print(f"  section filled:         {sections_filled}/{len(all_chunks)} ({100*sections_filled//len(all_chunks)}%)")
print(f"  heading_hierarchy:      {hierarchies_filled}/{len(all_chunks)} ({100*hierarchies_filled//len(all_chunks)}%)")
print()

# Distribution by doc_type
from collections import Counter
type_dist = Counter(c.metadata["doc_type"] for c in all_chunks)
print("📁 Chunks by doc_type")
print("=" * 50)
for doc_type, count in type_dist.most_common():
    print(f"  {doc_type:<20} {count:>4} chunks")

📊 Chunk Quality Report
  Total chunks:           292
  Min chunk size:         99 chars
  Max chunk size:         1000 chars
  Avg chunk size:         501 chars
  Median chunk size:      447 chars

📋 Metadata Completeness
  doc_title filled:       292/292 (100%)
  section filled:         292/292 (100%)
  heading_hierarchy:      292/292 (100%)

📁 Chunks by doc_type
  architecture          143 chunks
  feature               118 chunks
  operations             24 chunks
  index                   7 chunks


## Step 6 — Export Processed Chunks

Save chunks as JSON to `data/processed/` for downstream ingestion (embedding + vector store).
The JSON format preserves both `page_content` and the enriched `metadata` so the ingestion
pipeline (`app/rag/ingestion.py`) can load them directly.

In [22]:
# ── Export to JSON ────────────────────────────────────────────────────────────

def chunks_to_serializable(chunks: list[Document]) -> list[dict]:
    """Convert LangChain Documents to JSON-serializable dicts."""
    return [
        {
            "page_content": chunk.page_content,
            "metadata": chunk.metadata,
        }
        for chunk in chunks
    ]

output_path = PROCESSED_DIR / "chunks_enriched.json"
serializable = chunks_to_serializable(all_chunks)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(serializable, f, indent=2, ensure_ascii=False)

print(f"Exported {len(serializable)} chunks to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Exported 292 chunks to ../data/processed/chunks_enriched.json
File size: 306.4 KB


## Step 7 — Preview: How Retrieval Benefits From This Metadata

This cell shows how the enriched metadata enables **filtered retrieval** — something
that was impossible with the raw Unstructured output.

In [23]:
# ── Retrieval examples with enriched metadata ───────────────────────────────

print("🔍 Example: Filter chunks by doc_type='architecture'")
print("=" * 60)
arch_chunks = [c for c in all_chunks if c.metadata["doc_type"] == "architecture"]
print(f"  {len(arch_chunks)} chunks (out of {len(all_chunks)} total)\n")
for c in arch_chunks[:3]:
    print(f"  [{c.metadata['chunk_index']:>2}] {c.metadata['section']:<40} ({len(c.page_content)} chars)")

print(f"\n\n🔍 Example: Find all chunks under 'State Management' sections")
print("=" * 60)
state_chunks = [
    c for c in all_chunks
    if any("state" in h.lower() for h in c.metadata.get("heading_hierarchy", []))
]
for c in state_chunks[:5]:
    print(f"  {c.metadata['source']:<45} → {' > '.join(c.metadata['heading_hierarchy'])}")

print(f"\n\n🔍 Example: Reconstruct reading order for a document")
print("=" * 60)
overview_chunks = sorted(
    [c for c in all_chunks if "overview.md" in c.metadata["source"]],
    key=lambda c: c.metadata["chunk_index"],
)
for c in overview_chunks[:8]:
    print(f"  chunk[{c.metadata['chunk_index']:>2}]  section: {c.metadata['section'][:50]}")

print(f"\n\n💡 In your RAG chain, you can now do:")
print("""
    # FAISS with metadata filtering (via self-query or manual filter):
    retriever = vectorstore.as_retriever(
        search_kwargs={
            "k": 5,
            "filter": {"doc_type": "architecture"},
        }
    )

    # Or use a SelfQueryRetriever to let the LLM decide filters:
    from langchain.retrievers import SelfQueryRetriever
    metadata_field_info = [
        AttributeInfo(name="doc_type", description="Document category", type="string"),
        AttributeInfo(name="section", description="Section heading", type="string"),
        AttributeInfo(name="doc_title", description="Document title", type="string"),
    ]
""")

🔍 Example: Filter chunks by doc_type='architecture'
  143 chunks (out of 292 total)

  [ 0] Code Quality & Style Guide for Developers (130 chars)
  [ 1] Overview                                 (268 chars)
  [ 2] Strict Typing                            (506 chars)


🔍 Example: Find all chunks under 'State Management' sections
  architecture/code-quality.md                  → Code Quality & Style Guide for Developers > State Management Rules > What Goes Where
  architecture/code-quality.md                  → Code Quality & Style Guide for Developers > State Management Rules > Anti-Patterns
  architecture/overview.md                      → AMS Admin Tool — Architecture Documentation > State Management Strategy > When to Use What
  architecture/overview.md                      → AMS Admin Tool — Architecture Documentation > State Management Strategy > Redux Slices
  architecture/overview.md                      → AMS Admin Tool — Architecture Documentation > State Management Strategy > T